# 01 — RQ1: How widely do AI repos adopt security tools?

RQ1 reports the **adoption rate** of each security tool / category /
language slice inside the **AI cohort** (n=2,803 repos). It is a
descriptive RQ — there is no comparison to controls here. RQ2
(notebook 02) is the comparative RQ.

All numbers come from `analysis/tables/rq1_*.csv` and
`rq1_headline.json`. This notebook does not fit any models.

## Provenance

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

REPO_ROOT = (
    Path(__file__).resolve().parents[2]
    if "__file__" in globals()
    else Path.cwd().parents[1]
)
TABLES = REPO_ROOT / "analysis" / "tables"
FIGURES = REPO_ROOT / "analysis" / "figures"


class MissingArtifact(FileNotFoundError):
    pass


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise MissingArtifact(f"missing canonical CSV: {path}")
    return pd.read_csv(path)


def load_json(path: Path) -> dict:
    if not path.exists():
        raise MissingArtifact(f"missing canonical JSON: {path}")
    return json.loads(path.read_text())


prov = load_json(TABLES / "rq1_provenance.json")
print(f"run_id            : {prov['run_id']}")
print(f"aidev_dataset_sha : {prov['aidev_dataset_sha']}")
print(f"compute_script    : {prov['compute_script']}")
print(f"generated_at_utc  : {prov['generated_at_utc']}")
print()
print("configs hashes:")
for name, info in prov["configs_hashes"].items():
    print(f"  {name:32s}  {info['version_header']}  sha256={info['sha256'][:16]}…")

run_id            : 2026-04-25
aidev_dataset_sha : v3
compute_script    : analysis/scripts/rq1_compute.py
generated_at_utc  : 2026-04-26T19:21:59+00:00

configs hashes:
  agent_fingerprints.yaml           version: 2026-04-25  sha256=5c5a03502c2e9121…
  security_bots.txt                 version: 2026-04-26  sha256=edbd29526cce9e97…
  security_patterns.yaml            version: 2026-04-26  sha256=12c66325bfafe6bb…
  tools.yaml                        version: 2026-04-25  sha256=6d2df0bc3bba6117…


## Headline numbers

In [2]:
headline = load_json(TABLES / "rq1_headline.json")
pd.Series(
    {
        "n_AI_repos": headline["n_ai_repos"],
        "any_tool_adoption_pct": headline["any_tool_pct"],
        "top_category": f"{headline['top_category']['category']} ({headline['top_category']['adoption_pct']}%)",
        "bottom_category": f"{headline['bottom_category']['category']} ({headline['bottom_category']['adoption_pct']}%)",
        "top_tool": f"{headline['top3_tools'][0]['tool']} ({headline['top3_tools'][0]['adoption_pct']}%)",
        "second_tool": f"{headline['top3_tools'][1]['tool']} ({headline['top3_tools'][1]['adoption_pct']}%)",
        "third_tool": f"{headline['top3_tools'][2]['tool']} ({headline['top3_tools'][2]['adoption_pct']}%)",
    }
)

n_AI_repos                               2803
any_tool_adoption_pct                   40.35
top_category                     sca (34.82%)
bottom_category              fuzzing (0.392%)
top_tool                 dependabot (29.718%)
second_tool                  codeql (16.696%)
third_tool                  renovate (5.316%)
dtype: object

## Adoption by tool (sorted, descending)

Three tools dominate (`dependabot`, `codeql`, `renovate`); the
remainder all sit under 5%. The long tail is real: 9 of 19 tools have
adoption < 1%.

In [3]:
by_tool = load_csv(TABLES / "rq1_adoption_by_tool.csv")
by_tool

,tool,n_ai_repos,n_configured,adoption_rate,adoption_pct
0,dependabot,2803,833,0.297182,29.718
1,codeql,2803,468,0.166964,16.696
2,renovate,2803,149,0.053157,5.316
3,ossf_scorecard,2803,81,0.028898,2.890
4,harden_runner,2803,47,0.016768,1.677
5,trivy,2803,46,0.016411,1.641
6,sonarqube,2803,22,0.007849,0.785
7,anchore,2803,19,0.006778,0.678
8,snyk,2803,19,0.006778,0.678
9,gitleaks,2803,17,0.006065,0.606


![Adoption by tool](../figures/rq1_adoption_by_tool.png)

## Adoption by category

In [4]:
by_category = load_csv(TABLES / "rq1_adoption_by_category.csv")
by_category

,category,n_ai_repos,n_configured,adoption_rate,adoption_pct
0,sca,2803,976,0.348198,34.820
1,sast,2803,490,0.174813,17.481
2,ci_hardening,2803,99,0.035319,3.532
3,secrets,2803,27,0.009633,0.963
4,fuzzing,2803,11,0.003924,0.392


![Adoption by category](../figures/rq1_adoption_by_category.png)

## Adoption by primary language (top 10 languages by repo count)

In [5]:
by_language = load_csv(TABLES / "rq1_adoption_by_language.csv")
by_language

,language,n_ai_repos,any_tool_pct,sast_pct,sca_pct,secrets_pct,fuzzing_pct,ci_hardening_pct
0,TypeScript,647,39.258,16.074,32.921,0.618,0.000,1.700
1,Python,529,39.319,16.635,33.081,1.890,0.000,2.647
2,Go,242,60.744,30.165,56.612,1.240,0.413,7.851
3,C#,220,48.182,23.182,39.545,0.000,0.000,5.000
4,JavaScript,190,25.789,9.474,21.579,1.579,0.000,1.579
5,Rust,159,49.057,10.692,46.541,0.629,1.258,6.289
6,C++,119,37.815,21.008,33.613,0.000,2.521,4.202
7,Java,86,50.000,30.233,43.023,2.326,0.000,2.326
8,PHP,69,31.884,10.145,28.986,1.449,0.000,1.449
9,C,61,37.705,31.148,24.590,0.000,8.197,9.836


![Adoption by language](../figures/rq1_adoption_by_language.png)

## Adoption by stars-bin

Adoption rises monotonically with repo popularity: the 1000+ stars
bin is roughly 1.6× the 100–199 bin on `any_tool`.

In [6]:
by_stars = load_csv(TABLES / "rq1_adoption_by_stars.csv")
by_stars

,stars_bin,n_ai_repos,any_tool_pct,sast_pct,sca_pct,secrets_pct,fuzzing_pct,ci_hardening_pct
0,100-199,641,31.669,13.417,26.989,0.624,0.156,1.404
1,200-499,685,34.161,12.409,30.511,0.584,0.146,3.212
2,500-999,408,39.706,16.912,33.824,0.980,0.000,4.167
3,1000+,1069,49.766,23.386,42.657,1.403,0.842,4.771


## Logit terms (per-category, AI-cohort only)

Each row is one term in the per-category logit
`configured ~ language_grp + owner_type_grp + log_stars +
repo_age_days + log_days_since_push`. The `secrets` and `fuzzing`
fits have many separated levels (NaN std-err, |coef| ≈ 25) and
should be read with the headline category-rate table, not the term
table.

In [7]:
logit = load_csv(TABLES / "rq1_logit_terms.csv")
logit.head(20)

,category,term,coef,std_err,p_value,or_,ci_lo,ci_hi,n_obs,note
0,sast,Intercept,1.621079,0.384144,2.443234e-05,5.058545,2.382549,10.740125,2803,NaN
1,sast,C(language_grp)[T.C#],0.508593,0.346667,1.423503e-01,1.662950,0.842938,3.280671,2803,NaN
2,sast,C(language_grp)[T.C++],0.730381,0.380050,5.463075e-02,2.075871,0.985601,4.372195,2803,NaN
3,sast,C(language_grp)[T.Dart],1.643849,0.823135,4.581891e-02,5.175050,1.031004,25.975789,2803,NaN
4,sast,C(language_grp)[T.Go],0.009988,0.335295,9.762365e-01,1.010038,0.523521,1.948683,2803,NaN
5,sast,C(language_grp)[T.HTML],1.217017,0.620429,4.981220e-02,3.377099,1.000999,11.393415,2803,NaN
6,sast,C(language_grp)[T.Java],0.185474,0.389009,6.335139e-01,1.203789,0.561598,2.580329,2803,NaN
7,sast,C(language_grp)[T.JavaScript],1.245515,0.389465,1.383766e-03,3.474725,1.619598,7.454761,2803,NaN
8,sast,C(language_grp)[T.Kotlin],1.233592,0.566255,2.936808e-02,3.433542,1.131736,10.416927,2803,NaN
9,sast,C(language_grp)[T.Other],0.906776,0.350962,9.775061e-03,2.476327,1.244711,4.926602,2803,NaN


### Try changing X

Change `head(20)` to `query("category == 'sast' and p_value < 0.05")`
to see the SAST terms with p < 0.05. The full table is in
`analysis/tables/rq1_logit_terms.csv`.